In [3]:
import requests
import pandas as pd

In [42]:
URL = "https://api.gdc.cancer.gov/files"
HEADERS = {"Content-Type": "application/json"}

# Base query parameters for the GDC API request
query_params = {
    "op": "and",
    "content": [
        {
            "op":"=",
            "content":{
                "field": "experimental_strategy",
                "value": "Methylation Array"
            }
        },
        {
            "op":"=",
            "content":{
                "field": "platform",
                "value": "illumina human methylation 450"
            }
        }
    ]
}

In [43]:
def query_facet(filters: dict, facet_field: str) -> list[dict]:
    payload = {
        "filters": filters,
        "facets": facet_field,
        "size": 0, # only returns counts, not actual file data
        "format": "json"
    }

    r = requests.post(URL, headers=HEADERS, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()["data"]["aggregations"][facet_field]["buckets"]




In [44]:
def get_projects() -> list[str]:
    buckets = query_facet(query_params, "cases.project.project_id")
    return sorted(b["key"] for b in buckets if b["key"].startswith("TCGA"))

projects = get_projects()
print(projects)

['TCGA-ACC', 'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL', 'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC', 'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG', 'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV', 'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC', 'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM', 'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM']


In [45]:
def get_sample_type_counts(project_id: str) -> dict[str, int]:
    project_filter = {
        "op": "and",
        "content": query_params["content"] + [
            {
                "op":"=",
                "content":{
                    "field": "cases.project.project_id",
                    "value": project_id
                }
            }
        ]
    }
    buckets = query_facet(project_filter, "cases.samples.sample_type")
    return {b["key"]: b["doc_count"] for b in buckets}

counts = get_sample_type_counts("TCGA-BRCA")
counts

{'primary tumor': 2379, 'solid tissue normal': 291, 'metastatic': 15}

In [ ]:
def main():
    print("Querying GDC for TCGA 450k methylation projects...")
    projects = get_projects()
    print(f"Found {len(projects)} projects: {projects}.\n")

    rows = []
    for project_id in projects:
        counts = get_sample_type_counts(project_id)
        total = sum(counts.values())
        counts["project_id"] = project_id
        counts["total_files"] = total
        counts["normal_pct"] = round(counts.get("solid tissue normal", 0) / total * 100) if total > 0 else 0
        rows.append(counts)

    print(rows)
    df = (
        pd.DataFrame(rows)
        .sort_values("solid tissue normal", ascending=False)
        .reset_index(drop=True)
    )

    print("\n--- Results (sorted by solid_normal count) ---]\n")
    print(df.to_string(index=False))

    out_path = "C:\\Users\\JoshK\\OneDrive\\Desktop\\ML_Projects\\dna_methylation_cancer_prediction\\DATA\\GDC\\tcga_methylation_counts.csv"
    df.to_csv(out_path, index=False)
    print(f"\nSaved results to {out_path}")

main()

Querying GDC for TCGA 450k methylation projects...
Found 33 projects: ['TCGA-ACC', 'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL', 'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC', 'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG', 'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV', 'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC', 'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM', 'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM'].

[{'primary tumor': 240, 'project_id': 'TCGA-ACC', 'total_files': 240, 'normal_pct': 0}, {'primary tumor': 1254, 'solid tissue normal': 63, 'metastatic': 3, 'project_id': 'TCGA-BLCA', 'total_files': 1320, 'normal_pct': 5}, {'primary tumor': 2379, 'solid tissue normal': 291, 'metastatic': 15, 'project_id': 'TCGA-BRCA', 'total_files': 2685, 'normal_pct': 11}, {'primary tumor': 921, 'solid tissue normal': 9, 'metastatic': 6, 'project_id': 'TCGA-CESC', 'total_files': 936, 'normal_pct': 1}, {'primary tumor